# Retrieval-Augmented Generation (RAG) for InvestIQ



In [1]:
!pip install -q -U langchain-text-splitters sentence-transformers huggingface_hub chromadb torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7

---
## 1. Load the NVIDIA 10-K

Load NVIDIA's 2026 10-K filing as the source document for the RAG pipeline.



### Define the source file

Set the path to the NVIDIA 10-K text file.


In [5]:
Nvidia = "/content/10-K_2026-02-25_0001045810-26-000021.txt"


### Read and inspect the raw filing

Read the full filing and inspect the beginning of the text. This helps us see the raw SEC/XBRL content before selecting useful sections.

In [6]:
from pathlib import Path

file_path = Path(Nvidia)

text = file_path.read_text(encoding="utf-8")

print("Total characters:", len(text))
print(text[:2000])

Total characters: 360462
nvda-20260125
0001045810
2026
FY
false
362
460
P1Y
P2Y
P1Y
P3Y
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
iso4217:USD
xbrli:shares
iso4217:USD
xbrli:shares
xbrli:pure
nvda:period
nvda:segment
0001045810
2025-01-27
2026-01-25
0001045810
2025-07-25
0001045810
2026-02-20
0001045810
2025-10-27
2026-01-25
0001045810
nvda:JohnO.DabiriMember
2025-10-27
2026-01-25
0001045810
nvda:JohnO.DabiriMember
2026-01-25
0001045810
nvda:ColetteM.KressMember
2025-10-27
2026-01-25
0001045810
nvda:ColetteM.KressMember
2026-01-25
0001045810
2024-01-29
2025-01-26
0001045810
2023-01-30
2024-01-28
0001045810
2026-01-25
0001045810
2025-01-26
0001045810
us-gaap:CommonStockMember
2023-01-29
0001045810
us-gaap:AdditionalPaidInCapitalMember
2023-01-29
0001045810
us-gaap:AccumulatedOtherComprehensiveIncomeMember
2023-01-29
0001045810
us-gaap:RetainedEarningsMember
2023-01-29
0001045810
2023-01-29
0001045810
us-gaap:RetainedEarningsMember
2023-01-30
2024-01-28
0001045810
us-gaap:Acc

## 2. Business Section

### Locate the Business section

Search for `Item 1. Business` and inspect the possible section positions before choosing the correct one.

In [7]:
matches = []

start = 0

while True:
    pos = text.find("Item 1. Business", start)
    if pos == -1:
        break

    matches.append(pos)
    start = pos + 1

print(matches)

[30509]


In [8]:
business_pos = 30509

print(text[business_pos : business_pos + 2500])

Item 1. Business
Our Company
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. NVIDIA is now a data center scale AI infrastructure company reshaping all industries.
Our technology stack includes the foundational NVIDIA CUDA development platform that runs on all NVIDIA GPUs, as well as hundreds of domain-specific software libraries, frameworks, algorithms, software development kits, or SDKs, and application programming interfaces, or APIs. This deep and broad software stack accelerates the performance and facilitates the deployment of NVIDIA accelerated computing for computationally intensive workloads such as artificial intelligence, or AI, model training and inference, data analytics, scientific computing, robotics, and 3D graphics, with vertical-specific optimizations to address industries ranging from healthcare and telecom to automotive and manufacturing.
Introduced with the Blackwell architecture, our data-center-scale offerings feat

In [9]:
risk_pos = text.find("Item 1A. Risk Factors", business_pos)

print(risk_pos)
print(text[risk_pos : risk_pos + 500])

71211
Item 1A. Risk Factors – Risks Related to Regulatory, Legal, Our Stock, and Other Matters” for a discussion of this potential impact.
Compliance with laws, rules, and regulations has not otherwise had a material effect upon our capital expenditures, results of operations, or competitive position and we do not currently anticipate material capital expenditures for environmental control facilities. Compliance with existing or future governmental regulations, including, but not limited to, those per


In [10]:
risk_matches = []

start = business_pos

while True:
    pos = text.find("Item 1A. Risk Factors", start)
    if pos == -1:
        break

    risk_matches.append(pos)
    start = pos + 1

print(risk_matches)

[71211, 72358, 78751, 204367, 211263, 237006]


In [11]:
for pos in risk_matches:
    print("\nPOSITION:", pos)
    print(text[pos : pos + 250])
    print("-" * 80)


POSITION: 71211
Item 1A. Risk Factors – Risks Related to Regulatory, Legal, Our Stock, and Other Matters” for a discussion of this potential impact.
Compliance with laws, rules, and regulations has not otherwise had a material effect upon our capital expenditures, r
--------------------------------------------------------------------------------

POSITION: 72358
Item 1A. Risk Factors” for a discussion of these potential impacts.
Human Capital Management
As of the end of fiscal year 2026, we had approximately 42,000 employees in 38 countries; 31,000 were engaged in research and development and 11,000 were eng
--------------------------------------------------------------------------------

POSITION: 78751
Item 1A. Risk Factors
The following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. The following risks could harm our business, financial condition, results of operations or reputation, whic
----------------------------------

### Extract the Business text

Use the confirmed start and end positions to extract the Business section from the full filing.


In [12]:
business_start = 30509
business_end = 78751

In [13]:
business_text = text[business_start:business_end]

print("Business section characters:", len(business_text))
print(business_text[:1000])

Business section characters: 48242
Item 1. Business
Our Company
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. NVIDIA is now a data center scale AI infrastructure company reshaping all industries.
Our technology stack includes the foundational NVIDIA CUDA development platform that runs on all NVIDIA GPUs, as well as hundreds of domain-specific software libraries, frameworks, algorithms, software development kits, or SDKs, and application programming interfaces, or APIs. This deep and broad software stack accelerates the performance and facilitates the deployment of NVIDIA accelerated computing for computationally intensive workloads such as artificial intelligence, or AI, model training and inference, data analytics, scientific computing, robotics, and 3D graphics, with vertical-specific optimizations to address industries ranging from healthcare and telecom to automotive and manufacturing.
Introduced with the Blackwell architecture, o

### Organize the extracted section

Store the Business text with a section label before chunking.

In [14]:
business_section = {
    "section": "Business",
    "text": business_text
}

print(business_section["section"])
print(len(business_section["text"]))

Business
48242


### Split the Business section into chunks

Use `RecursiveCharacterTextSplitter` with a chunk size of 3000 characters and 300 characters of overlap. Smaller chunks make the document easier to retrieve later.


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300
)

business_chunks = text_splitter.split_text(business_text)

print("Number of Business chunks:", len(business_chunks))
print("\nFirst chunk:\n")
print(business_chunks[0])

Number of Business chunks: 20

First chunk:

Item 1. Business
Our Company
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. NVIDIA is now a data center scale AI infrastructure company reshaping all industries.
Our technology stack includes the foundational NVIDIA CUDA development platform that runs on all NVIDIA GPUs, as well as hundreds of domain-specific software libraries, frameworks, algorithms, software development kits, or SDKs, and application programming interfaces, or APIs. This deep and broad software stack accelerates the performance and facilitates the deployment of NVIDIA accelerated computing for computationally intensive workloads such as artificial intelligence, or AI, model training and inference, data analytics, scientific computing, robotics, and 3D graphics, with vertical-specific optimizations to address industries ranging from healthcare and telecom to automotive and manufacturing.
Introduced with the Blackwell archi

### Inspect the chunks

Check chunk boundaries and sizes to make sure the split produced readable and meaningful pieces.

In [16]:
print("End of chunk 1:\n")
print(business_chunks[0][-500:])

print("\nStart of chunk 2:\n")
print(business_chunks[1][:500])

End of chunk 1:

ng cars that can perceive, understand and reason about the world. GPU-powered AI solutions are being developed by thousands of enterprises to deliver services and products that would have been immensely difficult or even impossible with traditional coding. Examples include generative AI, which can create new content such as text, code, images, audio, video, molecule structures, and recommendation systems; and agentic AI where systems of AI models work in concert to automatically complete a task.

Start of chunk 2:

NVIDIA has a platform strategy, bringing together hardware, systems, software, algorithms, libraries, AI models and training data sets, and services to create unique value for the markets we serve. While the computing requirements of these end markets are diverse, we address them with a unified underlying programmable architecture allowing us to support several multi-billion-dollar end markets with the same underlying technology by using a variety of softwar

In [17]:
for i, chunk in enumerate(business_chunks):
    print(i + 1, len(chunk))

1 2434
2 709
3 2858
4 2824
5 2767
6 2844
7 2959
8 2661
9 2046
10 2194
11 2197
12 2927
13 2719
14 2675
15 2369
16 2572
17 2428
18 2743
19 2697
20 2258


In [18]:
print(business_chunks[1])

NVIDIA has a platform strategy, bringing together hardware, systems, software, algorithms, libraries, AI models and training data sets, and services to create unique value for the markets we serve. While the computing requirements of these end markets are diverse, we address them with a unified underlying programmable architecture allowing us to support several multi-billion-dollar end markets with the same underlying technology by using a variety of software stacks developed either internally or by third-party developers and partners. The large and growing number of developers and installed base across our platforms strengthens our ecosystem and increases the value of our platform for our customers.


### Add Business metadata

Give each Business chunk a unique ID and metadata such as company, ticker, filing year, and section.


In [49]:
processed_chunks = []

for i, chunk in enumerate(business_chunks, start=1):
    chunk_id = f"NVDA_Business_{i:03d}"

    processed_chunks.append({
        "id": chunk_id,
        "text": chunk,
        "metadata": {
            "filing_year": 2026,
            "section": "Business",
            "company": "NVIDIA",
            "source": "SEC EDGAR",
            "ticker": "NVDA",
            "chunk_id": chunk_id,
            "filing_type": "10-K"
        }
    })

print(processed_chunks[0])

{'id': 'NVDA_Business_001', 'text': 'Item 1. Business\nOur Company\nNVIDIA pioneered accelerated computing to help solve the most challenging computational problems. NVIDIA is now a data center scale AI infrastructure company reshaping all industries.\nOur technology stack includes the foundational NVIDIA CUDA development platform that runs on all NVIDIA GPUs, as well as hundreds of domain-specific software libraries, frameworks, algorithms, software development kits, or SDKs, and application programming interfaces, or APIs. This deep and broad software stack accelerates the performance and facilitates the deployment of NVIDIA accelerated computing for computationally intensive workloads such as artificial intelligence, or AI, model training and inference, data analytics, scientific computing, robotics, and 3D graphics, with vertical-specific optimizations to address industries ranging from healthcare and telecom to automotive and manufacturing.\nIntroduced with the Blackwell architect

### Save a temporary Business chunk file

Save the processed Business chunks as JSON for inspection and backup.


In [20]:
import json

with open("nvda_business_chunks.json", "w", encoding="utf-8") as f:
    json.dump(processed_chunks, f, indent=2, ensure_ascii=False)

print("Saved:", len(processed_chunks), "chunks")

Saved: 20 chunks


## 3. Risk Factors

Extract and process NVIDIA's `Item 1A. Risk Factors` section.

### Locate and extract Risk Factors

Use the confirmed start position and find `Item 1B` as the end boundary of the Risk Factors section.

In [25]:
risk_start = 78751

In [26]:
risk_end = text.find("Item 1B. Unresolved Staff Comments", risk_start)

print(risk_end)
print(text[risk_end : risk_end + 500])

193668
Item 1B. Unresolved Staff Comments
None.
Item 1C. Cybersecurity
Risk management and strategy
We have in place certain infrastructure, systems, policies, and procedures that are designed to proactively prevent or reduce the impact of, and reactively address circumstances that arise when, events such as a cybersecurity incident occur. These include processes for assessing, identifying, and managing material risks from cybersecurity threats. Our information security management programs generally fo


In [27]:
risk_text = text[risk_start:risk_end]

print("Risk Factors characters:", len(risk_text))
print(risk_text[:1000])

Risk Factors characters: 114917
Item 1A. Risk Factors
The following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. The following risks could harm our business, financial condition, results of operations or reputation, which could cause our stock price to decline. Additional risks, trends and uncertainties not presently known to us or that we currently believe are immaterial may also harm our business, financial condition, results of operations or reputation.
Risk Factors Summary
Risks Related to Our Industry and Markets
•
Failure to meet the evolving needs of our industry and markets may adversely impact our financial results.
•
Competition could adversely impact our market share and financial results.
Risks Related to Demand, Supply, and Manufacturing
•
Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has led and could lead to mismatch

### Chunk the Risk Factors section

Split the extracted Risk Factors text using the same chunking settings as the Business section.

In [28]:
risk_chunks = text_splitter.split_text(risk_text)

print("Number of Risk Factor chunks:", len(risk_chunks))
print("\nFirst Risk Factors chunk:\n")
print(risk_chunks[0])

Number of Risk Factor chunks: 49

First Risk Factors chunk:

Item 1A. Risk Factors
The following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. The following risks could harm our business, financial condition, results of operations or reputation, which could cause our stock price to decline. Additional risks, trends and uncertainties not presently known to us or that we currently believe are immaterial may also harm our business, financial condition, results of operations or reputation.
Risk Factors Summary
Risks Related to Our Industry and Markets
•
Failure to meet the evolving needs of our industry and markets may adversely impact our financial results.
•
Competition could adversely impact our market share and financial results.
Risks Related to Demand, Supply, and Manufacturing
•
Long manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand accurately has l

### Add Risk Factors metadata

1.   List item
2.   List item



Add the Risk Factors chunks to the main `processed_chunks` collection with unique IDs and metadata.


In [29]:
for i, chunk in enumerate(risk_chunks, start=1):
    chunk_id = f"NVDA_Risk_Factors_{i:03d}"

    processed_chunks.append({
        "id": chunk_id,
        "text": chunk,
        "metadata": {
            "filing_year": 2026,
            "section": "Risk Factors",
            "company": "NVIDIA",
            "source": "SEC EDGAR",
            "ticker": "NVDA",
            "chunk_id": chunk_id,
            "filing_type": "10-K"
        }
    })

print("Total chunks so far:", len(processed_chunks))
print(processed_chunks[20])

Total chunks so far: 69
{'id': 'NVDA_Risk_Factors_001', 'text': 'Item 1A. Risk Factors\nThe following risk factors should be considered in addition to the other information in this Annual Report on Form 10-K. The following risks could harm our business, financial condition, results of operations or reputation, which could cause our stock price to decline. Additional risks, trends and uncertainties not presently known to us or that we currently believe are immaterial may also harm our business, financial condition, results of operations or reputation.\nRisk Factors Summary\nRisks Related to Our Industry and Markets\n•\nFailure to meet the evolving needs of our industry and markets may adversely impact our financial results.\n•\nCompetition could adversely impact our market share and financial results.\nRisks Related to Demand, Supply, and Manufacturing\n•\nLong manufacturing lead times and uncertain supply and capacity availability, combined with a failure to estimate customer demand ac

## 4. Management's Discussion and Analysis (MD&A)

### Locate the MD&A section

Search for `Item 7` and inspect the filing to identify the start of Management's Discussion and Analysis.


In [30]:
item7_matches = []

start = risk_end

while True:
    pos = text.find("Item 7.", start)
    if pos == -1:
        break

    item7_matches.append(pos)
    start = pos + 1

print(item7_matches)

[204146]


In [31]:
mda_start = 204146

print(text[mda_start : mda_start + 1200])

Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
The following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors,” our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold, or sell shares of our common stock.
Overview
Our Company and Our Businesses
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Fueled by the sustained demand for exceptional 3D graphics and the scale of the gaming market, NVIDIA has leveraged its GPU architecture to create platforms for scientific computing, AI, data science, autonomous vehicles, robotics, and digital twi

### Extract the MD&A text

Find `Item 7A` as the end boundary and extract the full MD&A section.


In [33]:
mda_end = text.find(
    "Item 7A. Quantitative and Qualitative Disclosures about Market Risk",
    mda_start
)

print(mda_end)
print(text[mda_end : mda_end + 500])

238301
Item 7A. Quantitative and Qualitative Disclosures about Market Risk
Investment and Interest Rate Risk
We are exposed to interest rate risk related to our fixed-rate investment portfolio and outstanding debt. The investment portfolio is managed consistent with our overall liquidity strategy in support of both working capital needs and growth of our businesses.
According to our sensitivity analysis on our investment portfolio, a decrease in the yield curve of 0.5% as of the end of fiscal year 2026


In [34]:
mda_text = text[mda_start:mda_end]

print("MD&A characters:", len(mda_text))
print(mda_text[:1000])

MD&A characters: 34155
Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
The following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors,” our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold, or sell shares of our common stock.
Overview
Our Company and Our Businesses
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Fueled by the sustained demand for exceptional 3D graphics and the scale of the gaming market, NVIDIA has leveraged its GPU architecture to create platforms for scientific computing, AI, data science, autonomous vehicles, ro

### Chunk and store MD&A

Split the MD&A section into smaller chunks using the same text splitter.


In [35]:
mda_chunks = text_splitter.split_text(mda_text)

print("Number of MD&A chunks:", len(mda_chunks))
print("\nFirst MD&A chunk:\n")
print(mda_chunks[0])

Number of MD&A chunks: 13

First MD&A chunk:

Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
The following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors,” our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold, or sell shares of our common stock.
Overview
Our Company and Our Businesses
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Fueled by the sustained demand for exceptional 3D graphics and the scale of the gaming market, NVIDIA has leveraged its GPU architecture to create platforms for scientific computing, AI, data science, 

### Add MD&A metadata

Add the MD&A chunks to the main processed collection with section-specific IDs and metadata.


In [36]:
for i, chunk in enumerate(mda_chunks, start=1):
    chunk_id = f"NVDA_MDA_{i:03d}"

    processed_chunks.append({
        "id": chunk_id,
        "text": chunk,
        "metadata": {
            "filing_year": 2026,
            "section": "MD&A",
            "company": "NVIDIA",
            "source": "SEC EDGAR",
            "ticker": "NVDA",
            "chunk_id": chunk_id,
            "filing_type": "10-K"
        }
    })

print("Total chunks so far:", len(processed_chunks))
print(processed_chunks[69])


Total chunks so far: 82
{'id': 'NVDA_MDA_001', 'text': 'Item 7. Management\'s Discussion and Analysis of Financial Condition and Results of Operations\nThe following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors,” our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold, or sell shares of our common stock.\nOverview\nOur Company and Our Businesses\nNVIDIA pioneered accelerated computing to help solve the most challenging computational problems. Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Fueled by the sustained demand for exceptional 3D graphics and the scale of the gaming market, NVIDIA has leveraged its GPU architecture to create platforms for scientific computing, AI,

## 5. Financial Statements

### Locate the financial statements

The `Item 8` heading points to the financial statements later in the filing. We search for the actual consolidated statements instead of using the first heading blindly.

In [38]:
item8_matches = []

start = mda_end

while True:
    pos = text.find("Item 8.", start)
    if pos == -1:
        break

    item8_matches.append(pos)
    start = pos + 1

print(item8_matches)

[]


In [39]:
item8_pos = text.find("Financial Statements and Supplementary Data", mda_end)

print(item8_pos)
print(text[item8_pos - 20 : item8_pos + 500])

242563
nformation.
Item 8. Financial Statements and Supplementary Data
The information required by this Item is set forth in our Consolidated Financial Statements and Notes thereto included in this Annual Report on Form 10-K.
Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure
None.
Item 9A. Controls and Procedures
Controls and Procedures
Disclosure Controls and Procedures
Based on their evaluation as of January 25, 2026, our management, including our Chief Executive Officer and Ch


### Find the real statement start

Search for occurrences of `Consolidated Statements` and inspect them to distinguish the table of contents from the actual financial statements.

In [40]:
financial_matches = []

start = item8_pos

while True:
    pos = text.find("Consolidated Statements", start)
    if pos == -1:
        break

    financial_matches.append(pos)
    start = pos + 1

print(financial_matches)

[251946, 252071, 252291, 252430, 262265, 263280, 265481, 267361, 278860, 282910, 283489, 293222, 294070, 294492, 296748, 309878]


In [41]:
for pos in financial_matches:
    print("\nPOSITION:", pos)
    print(text[pos : pos + 250])
    print("-" * 80)


POSITION: 251946
Consolidated Statements of Income for the years ended January 2
5
, 202
6
, January 2
6
, 202
5
, and January 2
8
, 202
4
51
Consolidated Statements of Comprehensive Income for the years ended January 2
5
, 202
6
, January 2
6
, 202
5
, and January 2
--------------------------------------------------------------------------------

POSITION: 252071
Consolidated Statements of Comprehensive Income for the years ended January 2
5
, 202
6
, January 2
6
, 202
5
, and January 2
8
, 202
4
52
Consolidated Balance Sheets as of
January 2
5
, 202
6
and January 2
6
, 202
5
53
Consolidated Statements of Sha
--------------------------------------------------------------------------------

POSITION: 252291
Consolidated Statements of Shareholders’ Equity for the years ended January 2
5
, 202
6
, January 2
6
, 202
5
, and January 2
8
, 202
4
54
Consolidated Statements of Cash Flows for the years ended January 2
5
, 202
6
, January 2
6
, 202
5
, and Janua
-------------------------------

### Confirm the financial statement start

Use the confirmed position where the actual Consolidated Statements of Income begin.

In [42]:
financial_start = 262265

print(text[financial_start : financial_start + 1200])

Consolidated Statements of Income
(In millions, except per share data)
Year Ended
Jan 25, 2026
Jan 26, 2025
Jan 28, 2024
Revenue
$
215,938
$
130,497
$
60,922
Cost of revenue
62,475
32,639
16,621
Gross profit
153,463
97,858
44,301
Operating expenses
Research and development
18,497
12,914
8,675
Sales, general and administrative
4,579
3,491
2,654
Total operating expenses
23,076
16,405
11,329
Operating income
130,387
81,453
32,972
Interest income
2,300
1,786
866
Interest expense
(
259
)
(
247
)
(
257
)
Other income, net
9,022
1,034
237
Total other income, net
11,063
2,573
846
Income before income tax
141,450
84,026
33,818
Income tax expense
21,383
11,146
4,058
Net income
$
120,067
$
72,880
$
29,760
Net income per share:
Basic
$
4.93
$
2.97
$
1.21
Diluted
$
4.90
$
2.94
$
1.19
Weighted average shares used in per share computation:
Basic
24,359
24,555
24,690
Diluted
24,514
24,804
24,940
See accompanying Notes to the Consolidated Financial Statements.
51
Table of Contents
NVIDIA Corporation an

### Find the end boundary

Search for `Schedule II`, which appears after the main financial statements and notes, and use it as the stopping point.

In [43]:
schedule_matches = []

start = financial_start

while True:
    pos = text.find("Schedule II", start)
    if pos == -1:
        break

    schedule_matches.append(pos)
    start = pos + 1

print(schedule_matches)

[349930]


In [44]:
schedule_start = 349930

print(text[schedule_start : schedule_start + 800])

Schedule II – Valuation and Qualifying Accounts
Description
Balance at
Beginning of Period
Additions
Deductions
Balance at
End of Period
(In millions)
Fiscal year 2026
Allowance for doubtful accounts
$
4
$
—
(1)
$
—
(1)
$
4
Sales return allowance
$
82
$
188
(2)
$
(
100
)
(4)
$
170
Deferred tax valuation allowance
$
1,610
$
31
(3)
$
(
873
)
(3)
$
768
Fiscal year 2025
Allowance for doubtful accounts
$
4
$
—
(1)
$
—
(1)
$
4
Sales return allowance
$
109
$
151
(2)
$
(
178
)
(4)
$
82
Deferred tax valuation allowance
$
1,552
$
58
(3)
$
—
(3)
$
1,610
Fiscal year 2024
Allowance for doubtful accounts
$
4
$
—
(1)
$
—
(1)
$
4
Sales return allowance
$
26
$
213
(2)
$
(
130
)
(4)
$
109
Deferred tax valuation allowance
$
1,484
$
162
(3)
$
(
94
)
(3)
$
1,552
(1)
Additions represent either expense or acquir


### Find the end boundary

Search for `Schedule II`, which appears after the main financial statements and notes, and use it as the stopping point.

In [45]:
financial_end = 349930

financial_text = text[financial_start:financial_end]

print("Financial section characters:", len(financial_text))
print(financial_text[:1000])

Financial section characters: 87665
Consolidated Statements of Income
(In millions, except per share data)
Year Ended
Jan 25, 2026
Jan 26, 2025
Jan 28, 2024
Revenue
$
215,938
$
130,497
$
60,922
Cost of revenue
62,475
32,639
16,621
Gross profit
153,463
97,858
44,301
Operating expenses
Research and development
18,497
12,914
8,675
Sales, general and administrative
4,579
3,491
2,654
Total operating expenses
23,076
16,405
11,329
Operating income
130,387
81,453
32,972
Interest income
2,300
1,786
866
Interest expense
(
259
)
(
247
)
(
257
)
Other income, net
9,022
1,034
237
Total other income, net
11,063
2,573
846
Income before income tax
141,450
84,026
33,818
Income tax expense
21,383
11,146
4,058
Net income
$
120,067
$
72,880
$
29,760
Net income per share:
Basic
$
4.93
$
2.97
$
1.21
Diluted
$
4.90
$
2.94
$
1.19
Weighted average shares used in per share computation:
Basic
24,359
24,555
24,690
Diluted
24,514
24,804
24,940
See accompanying Notes to the Consolidated Financial Statements.
51
Tab

### Chunk the financial statements

Split the financial section into smaller searchable chunks using the same chunking configuration.


In [46]:
financial_chunks = text_splitter.split_text(financial_text)

print("Number of Financial chunks:", len(financial_chunks))
print("\nFirst Financial chunk:\n")
print(financial_chunks[0])

Number of Financial chunks: 34

First Financial chunk:

Consolidated Statements of Income
(In millions, except per share data)
Year Ended
Jan 25, 2026
Jan 26, 2025
Jan 28, 2024
Revenue
$
215,938
$
130,497
$
60,922
Cost of revenue
62,475
32,639
16,621
Gross profit
153,463
97,858
44,301
Operating expenses
Research and development
18,497
12,914
8,675
Sales, general and administrative
4,579
3,491
2,654
Total operating expenses
23,076
16,405
11,329
Operating income
130,387
81,453
32,972
Interest income
2,300
1,786
866
Interest expense
(
259
)
(
247
)
(
257
)
Other income, net
9,022
1,034
237
Total other income, net
11,063
2,573
846
Income before income tax
141,450
84,026
33,818
Income tax expense
21,383
11,146
4,058
Net income
$
120,067
$
72,880
$
29,760
Net income per share:
Basic
$
4.93
$
2.97
$
1.21
Diluted
$
4.90
$
2.94
$
1.19
Weighted average shares used in per share computation:
Basic
24,359
24,555
24,690
Diluted
24,514
24,804
24,940
See accompanying Notes to the Consolidated Financia

### Add Financial Statements metadata

Add the financial chunks to the main processed collection with unique IDs and metadata.


In [47]:
for i, chunk in enumerate(financial_chunks, start=1):
    chunk_id = f"NVDA_Financial_Statements_{i:03d}"

    processed_chunks.append({
        "id": chunk_id,
        "text": chunk,
        "metadata": {
            "filing_year": 2026,
            "section": "Financial Statements",
            "company": "NVIDIA",
            "source": "SEC EDGAR",
            "ticker": "NVDA",
            "chunk_id": chunk_id,
            "filing_type": "10-K"
        }
    })

print("Total chunks:", len(processed_chunks))
print(processed_chunks[82])

Total chunks: 116
{'id': 'NVDA_Financial_Statements_001', 'text': "Consolidated Statements of Income\n(In millions, except per share data)\nYear Ended\nJan 25, 2026\nJan 26, 2025\nJan 28, 2024\nRevenue\n$\n215,938\n$\n130,497\n$\n60,922\nCost of revenue\n62,475\n32,639\n16,621\nGross profit\n153,463\n97,858\n44,301\nOperating expenses\nResearch and development\n18,497\n12,914\n8,675\nSales, general and administrative\n4,579\n3,491\n2,654\nTotal operating expenses\n23,076\n16,405\n11,329\nOperating income\n130,387\n81,453\n32,972\nInterest income\n2,300\n1,786\n866\nInterest expense\n(\n259\n)\n(\n247\n)\n(\n257\n)\nOther income, net\n9,022\n1,034\n237\nTotal other income, net\n11,063\n2,573\n846\nIncome before income tax\n141,450\n84,026\n33,818\nIncome tax expense\n21,383\n11,146\n4,058\nNet income\n$\n120,067\n$\n72,880\n$\n29,760\nNet income per share:\nBasic\n$\n4.93\n$\n2.97\n$\n1.21\nDiluted\n$\n4.90\n$\n2.94\n$\n1.19\nWeighted average shares used in per share computation:\nBasic

## 6. Save the Processed Dataset

Save all Business, Risk Factors, MD&A, and Financial Statements chunks into one JSON file. This processed file will be used for embeddings and semantic retrieval in the next stage of the RAG pipeline.

## 7. Create Embeddings

We use the `all-MiniLM-L6-v2` sentence-transformer model to convert each text chunk into a numerical embedding vector.

These embeddings allow us to compare the meaning of the user's question with the meaning of the document chunks.

In [55]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedding_function = SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print("Embedding function ready")

Embedding function ready


## 8. Create the Chroma Vector Store

We create a Chroma collection to store the NVIDIA document chunks and their embeddings.

Cosine distance is used to compare the similarity between the question embedding and the stored document embeddings.

In [57]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="nvidia_10k",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"}
)

print("Chroma collection ready")

Chroma collection ready


In [58]:
print("Chunks currently in Chroma:", collection.count())

Chunks currently in Chroma: 20


In [59]:
print("Chunks currently in Chroma:", collection.count())
print("processed_chunks:", len(processed_chunks))

Chunks currently in Chroma: 20
processed_chunks: 20


### Reload the Final Processed Chunks

The current session only contains the 20 Business chunks. We reload the previously saved combined JSON file containing all processed sections.

In [60]:
import json

with open("nvda_processed_chunks.json", "r", encoding="utf-8") as f:
    processed_chunks = json.load(f)

print("Processed chunks loaded:", len(processed_chunks))

sections = set(chunk["metadata"]["section"] for chunk in processed_chunks)
print("Sections:", sections)

Processed chunks loaded: 116
Sections: {'Business', 'Risk Factors', 'Financial Statements', 'MD&A'}


### Reset the Chroma Collection

The current Chroma collection contains only the old 20 Business chunks.

We remove it and create a fresh collection before adding all 116 processed chunks.

In [61]:
client.delete_collection("nvidia_10k")

collection = client.create_collection(
    name="nvidia_10k",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"}
)

print("Fresh Chroma collection created")

Fresh Chroma collection created


In [62]:
print("Chunks currently in Chroma:", collection.count())

Chunks currently in Chroma: 0


## 9. Store All Chunks in Chroma

We add all 116 processed NVIDIA chunks to the Chroma collection.

Chroma uses the embedding function to convert each chunk into a vector and stores the text, ID, and metadata for semantic search.

In [63]:
collection.add(
    documents=[chunk["text"] for chunk in processed_chunks],
    ids=[chunk["id"] for chunk in processed_chunks],
    metadatas=[chunk["metadata"] for chunk in processed_chunks]
)

print("Chunks stored in Chroma:", collection.count())

Chunks stored in Chroma: 116


## 10. Semantic Search

We test the vector database by asking a question about NVIDIA.

Chroma converts the question into an embedding and compares it with the stored chunk embeddings to retrieve the most relevant passages.

In [64]:
question = "What are NVIDIA's main business areas?"

results = collection.query(
    query_texts=[question],
    n_results=3
)

print("Question:", question)

for i, document in enumerate(results["documents"][0], start=1):
    print(f"\n--- Result {i} ---")
    print(document[:1000])

Question: What are NVIDIA's main business areas?

--- Result 1 ---
Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
The following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors,” our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold, or sell shares of our common stock.
Overview
Our Company and Our Businesses
NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields. Fueled by the sustained demand for exceptional 3D graphics and the scale of the gaming market, NVIDIA has leveraged its GPU architecture to create platforms for scientific computin

### Inspect Retrieved Results

We inspect the metadata and distance values of the retrieved chunks.

A smaller cosine distance means the retrieved chunk is more similar to the question.

In [65]:
results = collection.query(
    query_texts=[question],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i in range(3):
    print(f"\n--- Result {i + 1} ---")
    print("Section:", results["metadatas"][0][i]["section"])
    print("Chunk ID:", results["metadatas"][0][i]["chunk_id"])
    print("Distance:", results["distances"][0][i])
    print("Text:", results["documents"][0][i][:500])


--- Result 1 ---
Section: MD&A
Chunk ID: NVDA_MDA_001
Distance: 0.32081329822540283
Text: Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations
The following discussion and analysis of our financial condition and results of operations should be read in conjunction with “Item 1A. Risk Factors,” our Consolidated Financial Statements and related Notes thereto, as well as other cautionary statements and risks described elsewhere in this Annual Report on Form 10-K, before deciding to purchase, hold, or sell shares of our common stock.
Overview
Our C

--- Result 2 ---
Section: Business
Chunk ID: NVDA_Business_002
Distance: 0.3603513240814209
Text: NVIDIA has a platform strategy, bringing together hardware, systems, software, algorithms, libraries, AI models and training data sets, and services to create unique value for the markets we serve. While the computing requirements of these end markets are diverse, we address them with a unified underlying pr

## 11. Generate an Answer with Retrieved Context

We now combine retrieval with a language model.

The most relevant chunks are added to the prompt as context. Qwen then generates an answer using only the retrieved information.

In [71]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = (
    "/content/drive/MyDrive/InvestIQ_finetuned_adapter_updated"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("InvestIQ fine-tuned adapter loaded")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


InvestIQ fine-tuned adapter loaded


In [70]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 12. Build the RAG Prompt

We combine the retrieved document chunks into a context block.

The model is instructed to answer only from the retrieved context. If the information is not available, it should say that clearly.

In [76]:
context = "\n\n".join(results["documents"][0])

messages = [
    {
        "role": "system",
        "content": (
            "Answer the question using only the provided context. "
            "If the answer is not available in the context, say "
            "'Information not available.' Keep the answer brief."
        )
    },
    {
        "role": "user",
        "content": f"""
Context:
{context}

Question:
{question}
"""
    }
]

print("RAG prompt ready")

RAG prompt ready


## 13. Generate the Final Answer

We pass the RAG prompt to Qwen and generate a short answer based only on the retrieved NVIDIA context.

In [77]:
text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer(
    [text_input],
    return_tensors="pt"
).to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=200
)

response = tokenizer.decode(
    generated_ids[0][model_inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("Question:", question)
print("\nAnswer:")
print(response)

Question: What are NVIDIA's main business areas?

Answer:
Compute & Networking and Graphics.
